# KCC Preprocessing Pipeline

**Dataset:** Kisan Call Centre (KCC) Q&A logs — Uttar Pradesh, 2020–2025  
**Source:** Open Government Data Platform India (`data.gov.in`, resource ID: `cef25fe2-9231-4128-8aec-2c948fedd43f`)  
**Raw size:** 3,123,028 records, 15 columns (~2.07 GB)  
**Downstream consumer:** MuRIL-based FAISS retrieval index (constructed in Milestone 3)

---

## Preprocessing Stages

| Step | Operation | Purpose |
|------|-----------|----------|
| 1 | Load data | Parse combined CSV; strip `QueryType` whitespace padding |
| 2 | Overview & quality check | Verify UP-only scope and year range |
| 3 | Agronomic filter | Two-stage: Category allowlist + QueryType exclusion |
| 4 | Missing value treatment | Drop null Q&A; impute Season; fill categorical defaults |
| 5 | Deduplication | Remove exact rows, then duplicate (Q, A, Crop) triples |
| 6 | Text cleaning & PII removal | Normalise, redact phones/emails, detect language |
| 7 | Metadata tagging | Attach structured retrieval metadata to each record |
| 8 | Chunking | Split Q&A pairs into 512-char chunks for embedding |
| 9 | Chunk preview | Sanity-check output structure and distribution |
| 10 | Save artifacts | Write cleaned CSV, chunked JSONL, metadata schema |

**Outputs:** `kcc_cleaned_all_crops.csv`, `kcc_chunks_rag.jsonl`, `kcc_chunks_sample_1000.jsonl`, `metadata_schema.json`


In [1]:
# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to run on Colab)
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import pandas as pd
import numpy as np
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Data

Loads `kcc_combined_2020_2025.csv` produced by `03_kcc_rag_eda.ipynb` from the local repo path `../data/raw/kcc/`.  
Output directories `../data/processed/kcc/` and `../data/final/kcc/` are created if absent.

**`QueryType` strip (applied here):** EDA revealed that raw `QueryType` values carry tab padding (e.g., `\tPlant Protection\t`). Stripping is applied immediately after load so all downstream filters and metadata fields operate on clean values.


In [3]:
print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to use Drive)
# KCC_PATH = "/content/drive/MyDrive/kcc_raw/"

# Local paths — adhering to repo structure (notebooks executed from notebooks/ directory)
KCC_PATH = "../data/raw/kcc/"
PROCESSED_PATH = "../data/processed/kcc/"
FINAL_PATH = "../data/final/kcc/"

Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)
Path(FINAL_PATH).mkdir(parents=True, exist_ok=True)

data_file = f"{KCC_PATH}kcc_combined_2020_2025.csv"
print(f"Loading data from: {data_file}")

if not Path(data_file).exists():
    raise FileNotFoundError(
        f"❌ Combined CSV not found at '{data_file}'. "
        "Please refer to 'data/README.md' for download instructions."
    )

df = pd.read_csv(data_file)
print(f"✅ Loaded {len(df):,} records")
print(f"Columns: {len(df.columns)}")

# Strip tab/whitespace padding from QueryType (raw values arrive as e.g. '\tPlant Protection\t')
if 'QueryType' in df.columns:
    df['QueryType'] = df['QueryType'].str.strip()

print("\nFirst 5 rows:")
display(df.head())

print("\nData Info:")
print(df.info())


STEP 1: LOADING DATA
Loading data from: ../data/raw/kcc/kcc_combined_2020_2025.csv
✅ Loaded 3,123,028 records
Columns: 15

First 5 rows:


,BlockName,Category,CreatedOn,Crop,DistrictName,KCCCallID,KccAns,QueryText,QueryType,Season,Sector,StateName,day,month,year
0,KAKWAN,Vegetables,2020-12-15T11:22:07.863,Onion,KANPUR CITY,3369596,श्रीमान जी आप प्याज की नर्सरी में कार्बेंडाजिम...,Information related to preventing yellowing in...,Plant Protection,NaN,HORTICULTURE,UTTAR PRADESH,15,12,2020
1,BALLIA KHERI,Medicinal and Aromatic Plants,2020-12-15T11:28:13.737,Stevia,SAHARANPUR,3369885,सर आप स्टीविया के पौधे प्राप्त करने के लिए सीम...,information about seed and planting material s...,Seeds and Planting Material,NaN,HORTICULTURE,UTTAR PRADESH,15,12,2020
2,MAHRAJGANJ,Pulses,2020-12-15T11:28:28.1,Lentil (Masur),MAHARAHGANJ,3369482,श्रीमान जी मसूर की फसल में आप कॉपर ऑक्सीक्लोरा...,Information about fungus control in Lentil cro...,Plant Protection,NaN,AGRICULTURE,UTTAR PRADESH,15,12,2020
3,DHAULANA,Oilseeds,2020-12-15T11:30:33.08,Mustard,HAPUR (PANCHSHEEL NAGAR),3369709,श्रीमान जी सल्फर 80% wdg @ 400 से 500 ग्राम प्...,Information about nutrient management in musta...,Nutrient Management,NaN,AGRICULTURE,UTTAR PRADESH,15,12,2020
4,KHEKRA,Cereals,2020-12-12T07:52:25,Wheat,BAGHPAT,3344435,"श्रीमान जी,12 दिसंबर में बूंदाबादी होने की संभ...",Information about weather forecast of District...,Weather,NaN,AGRICULTURE,UTTAR PRADESH,12,12,2020



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123028 entries, 0 to 3123027
Data columns (total 15 columns):
 #   Column        Dtype  
---  ------        -----  
 0   BlockName     object 
 1   Category      object 
 2   CreatedOn     object 
 3   Crop          object 
 4   DistrictName  object 
 5   KCCCallID     int64  
 6   KccAns        object 
 7   QueryText     object 
 8   QueryType     object 
 9   Season        float64
 10  Sector        object 
 11  StateName     object 
 12  day           int64  
 13  month         int64  
 14  year          int64  
dtypes: float64(1), int64(4), object(10)
memory usage: 357.4+ MB
None


## Step 2: Data Overview & Quality Check

Audits schema, data types, and scope assumptions established during EDA:

- **`StateName`:** Confirmed 100% `UTTAR PRADESH` — the UP filter was applied at download time. No records need dropping; the column is dropped in Step 4 as uninformative.
- **Year range:** Confirmed 2020–2025 by dataset construction. No temporal filter is required.
- **Crop distribution:** Printed for reference; no crop-level filtering is applied (team decision: all agronomic crops retained; crop specificity is handled at retrieval time via metadata).


In [4]:
print("\n" + "="*80)
print("STEP 2: DATA OVERVIEW & QUALITY CHECK")
print("="*80)

print("\n2.1 Columns in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n2.2 Data types:")
print(df.dtypes)

print("\n2.3 Basic statistics for numeric columns:")
print(df.describe())

print("\n2.4 Verifying filters...")
if 'StateName' in df.columns:
    states = df['StateName'].unique()
    print(f"  States in data: {states}")
    if 'UTTAR PRADESH' in states:
        print("  Data is filtered for Uttar Pradesh")
    else:
        print("  Data may not be filtered for UP")

if 'year' in df.columns:
    years = sorted(df['year'].unique())
    print(f"  Years in data: {years}")
    if min(years) >= 2020:
        print("  Data is filtered for years >= 2020")

if 'Crop' in df.columns:
    print("\n2.5 Crop distribution (top 10):")
    print(df['Crop'].value_counts().head(10))


STEP 2: DATA OVERVIEW & QUALITY CHECK

2.1 Columns in dataset:
   1. BlockName
   2. Category
   3. CreatedOn
   4. Crop
   5. DistrictName
   6. KCCCallID
   7. KccAns
   8. QueryText
   9. QueryType
  10. Season
  11. Sector
  12. StateName
  13. day
  14. month
  15. year

2.2 Data types:
BlockName        object
Category         object
CreatedOn        object
Crop             object
DistrictName     object
KCCCallID         int64
KccAns           object
QueryText        object
QueryType        object
Season          float64
Sector           object
StateName        object
day               int64
month             int64
year              int64
dtype: object

2.3 Basic statistics for numeric columns:
          KCCCallID  Season           day         month          year
count  3.123028e+06     0.0  3.123028e+06  3.123028e+06  3.123028e+06
mean   3.364789e+06     NaN  1.563869e+01  6.093402e+00  2.022317e+03
std    2.101280e+06     NaN  8.854211e+00  3.599894e+00  1.605579e+00
min    2.

## Step 3: Agronomic Filter

Two-stage filter to remove records with no agronomic advisory value. The stages target different columns because the dataset encodes crop category and query intent separately.

### Stage 3a — Category allowlist (`Category` column)

Retains records whose `Category` matches a named crop group: Cereals, Pulses, Oilseeds, Vegetables, Fruits, Millets, Sugar and Starch Crops, Medicinal and Aromatic Plants, Condiments and Spices, Fodder Crops, Flowers, Fiber Crops, Plantation Crops, Green Manure.

**Why exclude `Others`:** `Others` is a catch-all bucket accounting for 35.0% of raw data (1.09M records). EDA inspection shows PM-KISAN scheme queries, generic pest calls, and non-crop enquiries are routed here. Because no crop-category signal is present, these records cannot be reliably boosted or filtered at retrieval time and add noise to the embedding space.

### Stage 3b — QueryType exclusion (`QueryType` column)

Drops records whose `QueryType` is `Weather`, `Government Schemes`, `Crop Insurance`, `Credit`, or `Market Information`.

| `QueryType` | Share of raw data | Reason for exclusion |
|---|---|---|
| `Weather` | 33.5% (~1.04M records) | 781,352 records are the identical string *"Farmer asked query on Weather…"* — auto-logged, zero agronomic content; weather forecasts are time-stamped and stale |
| `Government Schemes` | 25.6% (~800k records) | Scheme eligibility and subsidy amounts change over time; 2020–2025 logs risk producing confidently incorrect answers (M1 hallucination risk); authoritative scheme content is sourced from current PDFs instead |
| `Crop Insurance` / `Credit` / `Market Information` | ~5% combined | Financial and market-price scope; outside the agronomic advisory domain |

> **Correction note:** The original notebook applied the exclusion list to the `Category` column, where `Weather` and `Government Schemes` do not appear. The filter was corrected to target `QueryType` (where these values actually reside).


In [5]:
print("\n" + "="*80)
print("STEP 3: EXHAUSTIVE FOUR-STAGE HYBRID AGRONOMIC FILTER")
print("="*80)

# Track initial count
initial_count = len(df)
print(f"Total input rows before agronomic filter: {initial_count:,}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. DEFINE ALLOWLISTS & EXCLUSION LISTS
# ─────────────────────────────────────────────────────────────────────────────
agri_categories = [
    'Cereals', 'Pulses', 'Oilseeds', 'Vegetables', 'Fruits',
    'Millets', 'Sugar and Starch Crops', 'Medicinal and Aromatic Plants',
    'Condiments and Spices', 'Fodder Crops', 'Flowers', 'Fiber Crops',
    'Plantation Crops', 'Green Manure',
]

exclude_query_types = [
    # Basic non-agronomic metadata
    'Government Schemes', 'Weather', 'Crop Insurance', 'Credit', 'Market Information',
    # Administrative & infrastructure topics uncovered in leakage audit
    'Agriculture Mechanization', 'Training and Exposure Visits', 'Power, Roads etc.', 'Soil Health Card'
]

# ─────────────────────────────────────────────────────────────────────────────
# 2. EXHAUSTIVE SCRIPT-SEPARATED REGEX PATTERNS
# ─────────────────────────────────────────────────────────────────────────────
# QueryText (Q): English, Romanized Hinglish & Devanagari patterns
qt_weather_scheme_regex = (
    r'\bweather\b|weather forecast|mausam|barish|baaris|badal|baadal|tapan|tappman|tapman|'
    r'monsoon|\brain\b|rainfall|cloud|humidity|cyclone|toofan|\bole\b|hailstorm|dhoop|kohra|\bfog\b|\bfrost\b|\bpala\b|'
    r'मौसम|बारिश|वर्षा|पूर्वानुमान|बादल|तापमान|मानसून|ओलावृष्टि|कोहरा|पाला|'
    r'beneficiary status|pm kisan|pm-kisan|pradhan mantri kisan|samman nidhi|kisan samman|'
    r'farmer asked price detail|\(modal price\):|mandi price|\bbhav\b|\bbhaav\b|rate of|price of|'
    r'मंडी भाव|मोडल प्राइस|प्रधानमंत्री किसान'
)

qt_admin_land_animal_regex = (
    r'khatauni|khasra|lekhpal|tehsildar|mutation|gata no|bhu-naksha|kisan credit card|kcc loan|bank branch|patwari|'
    r'kisaan call center|toll free|helpline|complaint|district agriculture officer|up krishi nideshak|adhikari se sampark|1800-|'
    r'solar pump|boring|tube well|tubewell|subsidy|subsidies|anudan|farm machinery bank|chc application|custom hiring|tractor|rotavator|electricity|'
    r'\bcow\b|\bcattle\b|\bgoat\b|\bpoultry\b|\bhen\b|\bdog\b|\bmilk\b|\bmastitis\b|dairy|veterinary|pashudhan|pashu|fish\b|fishery|thanaula|'
    r'खतौनी|खसरा|लेखपाल|तहसीलदार|दाखिल ख़ारिज|भू-नक्शा|पटवारी|टोल फ्री|हेल्पलाइन|शिकायत करें|जिला कृषि अधिकारी से संपर्क|'
    r'सब्सिडी|अनुदान|सोलर पम्प|बोरिंग|ट्यूबवेल|बिजली|गाय|भैंस|बकरी|मुर्गी|थनैला|दूध|पशु|मछली'
)

# KccAns (A): Devanagari Hindi & English patterns
ans_weather_scheme_regex = (
    r'मौसम विभाग|मौसम का पूर्वानुमान|मौसम ज्ञात हो रहा|मौसम साफ रहेगा|बारिश होने की|बारिश की संभावना|'
    r'वर्षा होने की|बादल छाए|बूंदाबांदी|तापमान|अधिकतम तापमान|न्यूनतम तापमान|मानसून|ओला|कोहरा|पाला|'
    r'weather forecast|meteorological department|rain forecast|cloudy sky|'
    r'प्रधानमंत्री किसान सम्मान|लाभार्थी स्थिति|मंडी भाव|मोडल प्राइस|modal price|\bquintal\b|'
    r'क्विंटल|रु/क्विंटल|रु\./क्विंटल|स्टेटस में'
)

ans_admin_land_animal_regex = (
    r'खतौनी|खसरा|लेखपाल|तहसीलदार|दाखिल ख़ारिज|दाखिल खारिज|भू-नक्शा|पटवारी|किसान क्रेडिट कार्ड|बैंक शाखा|'
    r'1800-|टोल फ्री|हेल्पलाइन|शिकायत करें|जिला कृषि अधिकारी से संपर्क|उप कृषि निदेशक कार्यालय|'
    r'सब्सिडी|अनुदान|सोलर पम्प|सोलर पंप|बोरिंग|ट्यूबवेल|बिजली|फार्म मशीनरी|कृषि यंत्रों|'
    r'गाय|भैंस|बकरी|मुर्गी|थनैला|दूध|पशु|मछली|veterinary|dairy'
)

# Helper for pattern matching across Q OR A
def _has_pattern(dataframe, qt_pattern, ans_pattern):
    return (
        dataframe['QueryText'].fillna('').str.contains(qt_pattern, case=False) |
        dataframe['KccAns'].fillna('').str.contains(ans_pattern, case=False)
    )

# Helper for animal ignore terms (protecting wild animal repellants & fodder)
def _has_animal_protection(dataframe):
    ignore_pat = r'fodder|chara|straw|भूसा|चारा|cowpea|नील गाय|नीलगाय|neelgai|blue bull'
    return _has_pattern(dataframe, ignore_pat, ignore_pat)

# ─────────────────────────────────────────────────────────────────────────────
# Stage 3a: Category Allowlist + Null Agronomic Recovery
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Stage 3a: Category Allowlist & Null Metadata Recovery ---")
if 'Category' in df.columns:
    std_agri_mask = df['Category'].str.contains('|'.join(agri_categories), case=False, na=False)
    
    # Recover pure agronomic records logged under Category: Null where Crop is valid & text is clean
    known_crops_mask = (df['Crop'].notna()) & (~df['Crop'].isin(['Others', 'Unknown', 'None', '0']))
    recovery_mask = (
        df['Category'].isna() &
        known_crops_mask &
        (~_has_pattern(df, qt_weather_scheme_regex, ans_weather_scheme_regex)) &
        (~(_has_pattern(df, qt_admin_land_animal_regex, ans_admin_land_animal_regex) & ~_has_animal_protection(df)))
    )
    
    combined_mask = std_agri_mask | recovery_mask
    df_filtered = df[combined_mask].copy()
    print(f"  Standard category match : {std_agri_mask.sum():,} records")
    print(f"  Recovered from Null Cat : {recovery_mask.sum():,} records")
    print(f"  Total passing Stage 3a  : {len(df_filtered):,} ({len(df_filtered)/initial_count*100:.1f}%)")
else:
    df_filtered = df.copy()

# ─────────────────────────────────────────────────────────────────────────────
# Stage 3b: Expanded Metadata QueryType Exclusion (Safe for Nulls)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Stage 3b: Expanded Metadata QueryType Exclusion ---")
if 'QueryType' in df_filtered.columns:
    before_3b = len(df_filtered)
    exclude_qt_mask = df_filtered['QueryType'].str.contains(
        '|'.join(exclude_query_types), case=False, na=False
    )
    df_filtered = df_filtered[~exclude_qt_mask].copy()
    print(f"  Removed by QueryType metadata: {before_3b - len(df_filtered):,} records")
    print(f"  Total passing Stage 3b     : {len(df_filtered):,} ({len(df_filtered)/initial_count*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# Stage 3c: Language-Aware Weather & Scheme Scrubbing (~240k Scrub)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Stage 3c: Language-Aware Weather & Scheme Scrubbing ---")
before_3c = len(df_filtered)
drop_3c_mask = _has_pattern(df_filtered, qt_weather_scheme_regex, ans_weather_scheme_regex)
df_filtered = df_filtered[~drop_3c_mask].copy()
print(f"  Removed by weather/scheme text scrub: {before_3c - len(df_filtered):,} mislabeled records")
print(f"  Total passing Stage 3c              : {len(df_filtered):,} ({len(df_filtered)/initial_count*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# Stage 3d: Exhaustive Non-Agronomic Administrative, Legal & Animal Scrubbing (~26.8k Scrub)
# ─────────────────────────────────────────────────────────────────────────────
print("\n--- Stage 3d: Non-Agronomic Administrative, Legal & Animal Scrubbing ---")
before_3d = len(df_filtered)
candidate_3d_mask = _has_pattern(df_filtered, qt_admin_land_animal_regex, ans_admin_land_animal_regex)
protected_mask = _has_animal_protection(df_filtered)
drop_3d_mask = candidate_3d_mask & (~protected_mask)

df_filtered = df_filtered[~drop_3d_mask].copy()
print(f"  Removed by admin/legal/animal scrub : {before_3d - len(df_filtered):,} mislabeled records")
print(f"  FINAL AGRONOMIC RAG CORPUS          : {len(df_filtered):,} records ({len(df_filtered)/initial_count*100:.1f}% of raw corpus)")


STEP 3: EXHAUSTIVE FOUR-STAGE HYBRID AGRONOMIC FILTER
Total input rows before agronomic filter: 3,123,028

--- Stage 3a: Category Allowlist & Null Metadata Recovery ---
  Standard category match : 1,994,495 records
  Recovered from Null Cat : 0 records
  Total passing Stage 3a  : 1,994,495 (63.9%)

--- Stage 3b: Expanded Metadata QueryType Exclusion ---
  Removed by QueryType metadata: 889,917 records
  Total passing Stage 3b     : 1,104,578 (35.4%)

--- Stage 3c: Language-Aware Weather & Scheme Scrubbing ---
  Removed by weather/scheme text scrub: 206,433 mislabeled records
  Total passing Stage 3c              : 898,145 (28.8%)

--- Stage 3d: Non-Agronomic Administrative, Legal & Animal Scrubbing ---
  Removed by admin/legal/animal scrub : 12,118 mislabeled records
  FINAL AGRONOMIC RAG CORPUS          : 886,027 records (28.4% of raw corpus)


## Step 4: Missing Value Treatment

| Column | Strategy | Rationale |
|--------|----------|-----------|
| `QueryText` | Drop record | Query is the retrieval anchor; a missing query produces an unembeddable chunk |
| `KccAns` | Drop record | Answer is the retrieved content; a missing answer has no RAG value |
| `Season` | Infer from `month` (vectorised: Jun–Oct → Kharif, Nov–Mar → Rabi, Apr–May → Zaid) | `Season` column is 100% null in the raw data (confirmed by EDA); `month` is always populated |
| `Crop` | Fill `"Unknown"` | All crops are retained; `Unknown` is a valid metadata value for general-purpose queries |
| `DistrictName` | Fill `"Unknown"` | District is a retrieval filter; `Unknown` enables graceful fallback |
| `QueryType` | Fill `"Other"` | Residual nulls post-filter; `Other` is a non-informative but non-breaking default |

**Columns dropped as uninformative:** `CreatedOn` (superseded by `year`/`month`/`day`), `Sector` (constant: Agriculture), `StateName` (constant: UTTAR PRADESH), `KCCCallID` (call-log identifier, not needed for retrieval), `day`, `BlockName` (sub-district granularity not used in the metadata schema).


In [6]:
print("\n" + "="*80)
print("STEP 4: CHECKING & HANDLING MISSING VALUES")
print("="*80)

print("\nMissing values count per column:")
missing_counts = df_filtered.isnull().sum()
missing_pct = (missing_counts / len(df_filtered)) * 100

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_%': missing_pct.values
}).sort_values('Missing_%', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0].to_string(index=False))

print("\n" + "="*60)
print("Summary:")
print("="*60)
print(f"Total records: {len(df_filtered):,}")
print(f"Columns with missing values: {(missing_counts > 0).sum()}")
print(f"Total missing cells: {missing_counts.sum():,}")

print("\n" + "="*60)
print("Dropping Unnecessary Columns...")
print("="*60)

columns_to_drop = [
    'CreatedOn',      # Redundant (year/month/day exist)
    'Sector',         # Only Agriculture
    'StateName',      # All records are Uttar Pradesh
    'KCCCallID',      # Not needed for RAG
    'day',            # Not needed
    'BlockName'       # We have district
]

# Drop columns that exist
columns_to_drop = [col for col in columns_to_drop if col in df_filtered.columns]
df_filtered = df_filtered.drop(columns=columns_to_drop)

print(f"Dropped columns: {columns_to_drop}")
print(f"Remaining columns: {df_filtered.columns.tolist()}")

print("\n" + "="*60)
print("Handling Missing Values...")
print("="*60)


print("\n1. Critical Fields (QueryText & KccAns):")
print("-" * 40)

if 'QueryText' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['QueryText'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing QueryText")

if 'KccAns' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['KccAns'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing KccAns")

print(f"  Remaining records: {len(df_filtered):,}")

print("\n2. Crop Column (fill with 'Unknown'):")
print("-" * 40)

if 'Crop' in df_filtered.columns:
    missing_crop = df_filtered['Crop'].isna().sum()
    if missing_crop > 0:
        df_filtered['Crop'] = df_filtered['Crop'].fillna('Unknown')
        print(f"  Filled {missing_crop:,} missing Crop with 'Unknown'")
    else:
        print("  No missing Crop values found")

print("\n3. District Column (fill with 'Unknown'):")
print("-" * 40)

if 'DistrictName' in df_filtered.columns:
    missing_district = df_filtered['DistrictName'].isna().sum()
    if missing_district > 0:
        df_filtered['DistrictName'] = df_filtered['DistrictName'].fillna('Unknown')
        print(f"  Filled {missing_district:,} missing District with 'Unknown'")
    else:
        print("  No missing District values found")


print("\n4. Season Column (infer from month):")
print("-" * 40)

if 'Season' in df_filtered.columns and 'month' in df_filtered.columns:
    before = df_filtered['Season'].isna().sum()
    # Vectorised month → season mapping (Season is 100% null per EDA, so always inferred)
    def _month_to_season(m):
        if m in [6, 7, 8, 9, 10]:   return 'Kharif'
        elif m in [11, 12, 1, 2, 3]: return 'Rabi'
        else:                         return 'Zaid'
    df_filtered['Season'] = df_filtered['month'].map(_month_to_season).fillna('Unknown')
    after = df_filtered['Season'].isna().sum()
    print(f"  Inferred Season from month: {before - after:,} records filled")
    print(f"  Remaining missing: {after:,}")
elif 'Season' in df_filtered.columns:
    before = df_filtered['Season'].isna().sum()
    df_filtered['Season'] = df_filtered['Season'].fillna('Unknown')
    print(f"  Filled {before:,} missing Season with 'Unknown'")


print("\n5. QueryType Column (fill with 'Other'):")
print("-" * 40)

if 'QueryType' in df_filtered.columns:
    missing_qtype = df_filtered['QueryType'].isna().sum()
    if missing_qtype > 0:
        df_filtered['QueryType'] = df_filtered['QueryType'].fillna('Other')
        print(f"  Filled {missing_qtype:,} missing QueryType with 'Other'")
    else:
        print("  No missing QueryType values found")

if 'month' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['month'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing month")

if 'year' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['year'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing year")

print("\n" + "="*60)
print("Verification - No Critical Missing Values Should Remain")
print("="*60)

critical_fields = ['QueryText', 'KccAns', 'Crop', 'DistrictName']
for field in critical_fields:
    if field in df_filtered.columns:
        missing = df_filtered[field].isnull().sum()
        status = "OK" if missing == 0 else f"WARNING: {missing} missing"
        print(f"  {field}: {status}")

print("\n" + "="*60)
print("Final Summary")
print("="*60)
print(f"Records after handling missing values: {len(df_filtered):,}")
print(f"Columns remaining: {df_filtered.columns.tolist()}")


STEP 4: CHECKING & HANDLING MISSING VALUES

Missing values count per column:
      Column  Missing_Count  Missing_%
      Season         886027 100.000000
   BlockName            133   0.015011
      KccAns             82   0.009255
DistrictName              5   0.000564
        Crop              3   0.000339
   QueryType              3   0.000339
      Sector              3   0.000339

Summary:
Total records: 886,027
Columns with missing values: 7
Total missing cells: 886,256

Dropping Unnecessary Columns...
Dropped columns: ['CreatedOn', 'Sector', 'StateName', 'KCCCallID', 'day', 'BlockName']
Remaining columns: ['Category', 'Crop', 'DistrictName', 'KccAns', 'QueryText', 'QueryType', 'Season', 'month', 'year']

Handling Missing Values...

1. Critical Fields (QueryText & KccAns):
----------------------------------------
  Removed 0 records with missing QueryText
  Removed 82 records with missing KccAns
  Remaining records: 885,945

2. Crop Column (fill with 'Unknown'):
---------------

In [7]:
print("\n" + "="*60)
print("Verification - No Missing Values Should Remain")
print("="*60)

remaining_missing = df_filtered.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("✅ All missing values handled successfully!")
    print(f"   Final records: {len(df_filtered):,}")
else:
    print("⚠️ Still have missing values:")
    print(remaining_missing)


Verification - No Missing Values Should Remain
✅ All missing values handled successfully!
   Final records: 885,945


## Step 5: Deduplication

Two-pass deduplication:

1. **Exact row deduplication** (`drop_duplicates()` on all columns): removes byte-identical records introduced by annual file concatenation.
2. **Q&A-pair deduplication** (`subset=['QueryText', 'KccAns', 'Crop']`): removes records where the same question received the same answer for the same crop. Records where an identical query received *different* answers are intentionally preserved — diverse expert responses to the same question increase RAG recall and contextual coverage.

**EDA baseline:** 26.15% of (QueryText, KccAns, Crop) triples were exact duplicates in the raw corpus; 68.72% of QueryText values were non-unique (dominated by templated weather strings, already excluded in Step 3).


In [8]:
print("\n" + "="*80)
print("STEP 5: DUPLICATE REMOVAL")
print("="*80)

print(f"Records before deduplication: {len(df_filtered):,}")

print("\n" + "-"*60)
print("Removing exact duplicates (all columns)...")
print("-"*60)

before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(keep='first')
removed = before - len(df_filtered)
print(f"  Removed: {removed:,} exact duplicates")
print(f"  Remaining: {len(df_filtered):,}")

print("\n" + "-"*60)
print("Removing duplicate Q&A pairs for same crop...")
print("-"*60)

qa_cols = ['QueryText', 'KccAns', 'Crop']
qa_cols = [col for col in qa_cols if col in df_filtered.columns]

before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=qa_cols, keep='first')
removed = before - len(df_filtered)
print(f"  Removed: {removed:,} duplicate Q&A pairs for same crop")
print(f"  Remaining: {len(df_filtered):,}")

print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"  Records before dedup: {before:,}")
print(f"  Records after dedup:  {len(df_filtered):,}")
print(f"  Removed:              {before - len(df_filtered):,} duplicates")
print(f"  Reduction:            {(1 - len(df_filtered)/before)*100:.1f}%")


STEP 5: DUPLICATE REMOVAL
Records before deduplication: 885,945

------------------------------------------------------------
Removing exact duplicates (all columns)...
------------------------------------------------------------
  Removed: 21,213 exact duplicates
  Remaining: 864,732

------------------------------------------------------------
Removing duplicate Q&A pairs for same crop...
------------------------------------------------------------
  Removed: 154,111 duplicate Q&A pairs for same crop
  Remaining: 710,621

Summary
  Records before dedup: 864,732
  Records after dedup:  710,621
  Removed:              154,111 duplicates
  Reduction:            17.8%


## Step 6: Text Cleaning, PII Removal & Language Tagging

| Operation | Function | Detail |
|-----------|----------|--------|
| Whitespace normalisation | `clean_text()` | Collapses multi-space/newline sequences; strips encoding artefacts (`Ã`, `Â`, curly quotes) |
| Script filtering | `clean_text()` | Retains ASCII, Devanagari (U+0900–U+097F), Tamil, Telugu, Malayalam Unicode ranges; removes other non-alphanumeric characters |
| PII redaction | `remove_pii()` | 10-digit phone numbers → `[PHONE]`; email addresses → `[EMAIL]`; alphanumeric identifiers of 8+ chars → `[ID]` |
| Indic Unicode normalisation | `normalize_indic_text()` | Applies NFC composition; removes zero-width joiners and BOM characters that corrupt Devanagari rendering |
| Language detection | `detect_language()` | Heuristic: Devanagari block presence → `hi`; otherwise → `en`. EDA baseline: QueryText ~99.98% English/Romanised Hindi; KccAns ~98.8% Hindi (Devanagari) |
| Season inference | vectorised month map | Fills 100%-null `Season` column (see Step 4); included in this processing pass |

**PII note:** The KCC dataset contains call records. Redaction is applied before any artifact is written to disk.


In [9]:
print("\n" + "="*80)
print("STEP 6: TEXT CLEANING")
print("="*80)

def clean_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('Ã', '').replace('Â', '')
    text = text.replace('â', "'").replace('â', '"').replace('â', '"')
    text = re.sub(r'[^\w\s\u0900-\u097F\u0B80-\u0BFF\u0C00-\u0C7F\u0D00-\u0D7F.,!?\'"()-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_pii(text):
    if pd.isna(text) or text == '':
        return text
    text = re.sub(r'\b\d{10}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\b[A-Z0-9]{8,}\b', '[ID]', text)
    return text

def normalize_indic_text(text):
    if pd.isna(text) or text == '':
        return text
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
    return text

def detect_language(text):
    if pd.isna(text) or text == '' or len(str(text)) < 3:
        return 'unknown'
    text = str(text)
    if re.search(r'[\u0900-\u097F]', text):
        devanagari_chars = len(re.findall(r'[\u0900-\u097F]', text))
        total_chars = len(re.sub(r'\s', '', text))
        if total_chars > 0 and devanagari_chars / total_chars > 0.3:
            return 'hindi'
    english_chars = len(re.findall(r'[a-zA-Z]', text))
    total_chars = len(re.sub(r'\s', '', text))
    if total_chars > 0 and english_chars / total_chars > 0.5:
        return 'english'
    return 'mixed'

print("Cleaning QueryText and KccAns...")
df_filtered['cleaned_query'] = df_filtered['QueryText'].apply(clean_text)
df_filtered['cleaned_answer'] = df_filtered['KccAns'].apply(clean_text)

print("Removing PII...")
df_filtered['cleaned_query'] = df_filtered['cleaned_query'].apply(remove_pii)
df_filtered['cleaned_answer'] = df_filtered['cleaned_answer'].apply(remove_pii)

print("Normalizing Indian text...")
df_filtered['cleaned_query'] = df_filtered['cleaned_query'].apply(normalize_indic_text)
df_filtered['cleaned_answer'] = df_filtered['cleaned_answer'].apply(normalize_indic_text)

print("Detecting languages...")
df_filtered['query_lang'] = df_filtered['cleaned_query'].apply(detect_language)
df_filtered['answer_lang'] = df_filtered['cleaned_answer'].apply(detect_language)

print("\nLanguage distribution:")
print("  Query languages:")
for lang, count in df_filtered['query_lang'].value_counts().items():
    pct = count / len(df_filtered) * 100
    print(f"    {lang}: {count:,} ({pct:.1f}%)")

print("\nRemoving records with very short text...")
before = len(df_filtered)
df_filtered = df_filtered[
    (df_filtered['cleaned_query'].str.len() > 5) |
    (df_filtered['cleaned_answer'].str.len() > 5)
]
print(f"  Removed: {before - len(df_filtered):,} records")
print(f"  Remaining: {len(df_filtered):,}")


STEP 6: TEXT CLEANING
Cleaning QueryText and KccAns...
Removing PII...
Normalizing Indian text...
Detecting languages...

Language distribution:
  Query languages:
    english: 710,402 (100.0%)
    mixed: 212 (0.0%)
    unknown: 7 (0.0%)

Removing records with very short text...
  Removed: 4 records
  Remaining: 710,617


## Step 7: Metadata Tagging

Attaches a structured `metadata` dict to each record. This dict is embedded verbatim into each JSONL chunk and consumed by the FAISS retrieval pipeline for filtered search in Milestone 3.

| Field | Source column | Usage at retrieval time |
|---|---|---|
| `crop` | `Crop` | Filter retrieved chunks to a specific crop |
| `district` | `DistrictName` | Restrict retrieval to UP district-specific advisories |
| `block` | `BlockName` | Sub-district granularity (best-effort; frequently `Unknown`) |
| `season` | `Season` (inferred in Step 6) | Boost Kharif/Rabi-relevant chunks for seasonal queries |
| `query_type` | `QueryType` | Distinguish advisory, plant protection, nutrient management |
| `category` | `Category` | Crop group classification |
| `year` / `month` | `year`, `month` | Enable recency weighting or staleness filtering |
| `language` | `query_lang` (detected in Step 6) | Cross-lingual routing; all KccAns are Hindi |

The `metadata_schema` dict (initialised here, finalised in Step 10) documents chunk configuration, embedding model, and RAG retrieval tier thresholds for the Milestone 3 handoff.


In [10]:
print("\n" + "="*80)
print("STEP 7: METADATA TAGGING")
print("="*80)

print("Creating metadata columns...")
df_filtered['metadata'] = df_filtered.apply(lambda row: {
    'crop': row.get('Crop', 'unknown'),
    'district': row.get('DistrictName', 'unknown'),
    'block': row.get('BlockName', 'unknown'),
    'season': row.get('Season', 'unknown'),
    'query_type': row.get('QueryType', 'other'),
    'category': row.get('Category', 'other'),
    'year': int(row.get('year', 0)) if pd.notna(row.get('year')) else 0,
    'month': int(row.get('month', 0)) if pd.notna(row.get('month')) else 0,
    'language': row.get('query_lang', 'unknown')
}, axis=1)

print("Metadata sample:")
print(json.dumps(df_filtered['metadata'].iloc[0], indent=2, ensure_ascii=False))

# Build metadata schema dict (used in the save step)
metadata_schema = {
    'dataset': 'KCC Q&A Logs — Uttar Pradesh 2020–2025',
    'source': 'data.gov.in API (resource: cef25fe2-9231-4128-8aec-2c948fedd43f)',
    'chunk_config': {
        'chunk_size_chars': 512,
        'overlap_chars': 50,
        'format': 'Question: {query}\nAnswer: {answer}',
        'split_strategy': 'sentence boundaries (.!?)'
    },
    'embedding_model': 'MuRIL (google/muril-base-cased)',
    'metadata_fields': list(df_filtered['metadata'].iloc[0].keys()),
    'total_records': None,   # filled in save step
    'total_chunks': None,    # filled in save step
    'language_distribution': None,
    'crop_distribution': None,
    'query_type_distribution': None,
}
print("\nMetadata schema initialised.")


STEP 7: METADATA TAGGING
Creating metadata columns...
Metadata sample:
{
  "crop": "Onion",
  "district": "KANPUR CITY",
  "block": "unknown",
  "season": "Rabi",
  "query_type": "Plant Protection",
  "category": "Vegetables",
  "year": 2020,
  "month": 12,
  "language": "english"
}

Metadata schema initialised.


## Step 8: Chunking

Splits Q&A pairs into fixed-size text chunks suitable for dense vector embedding.

**Configuration:** `CHUNK_SIZE = 512 chars`, `OVERLAP = 50 chars`

**Chunk format:** `"Question: {query}\nAnswer: {answer}"` — the cross-lingual concatenation (English/Romanised-Hindi query + Devanagari-Hindi answer) is the unit MuRIL embeds. MuRIL is trained on 17 Indian languages and handles this bilingual structure natively.

**Split strategy:** If the combined Q&A text exceeds 512 chars, it is split at sentence boundaries (`.!?`) to avoid truncating mid-sentence. A 50-char overlap is applied between consecutive chunks to preserve context continuity at boundaries.

**Token vs character:** MuRIL's hard limit is 512 *tokens*, not characters. Devanagari Hindi encodes at approximately 2–3 chars per BPE subword token, so 512 chars ≈ 170–256 tokens — well within the limit for the vast majority of records (EDA: 98.9% of raw Q&A pairs fit within 512 chars as a single chunk). Outlier answers up to ~17 KB are handled by the sentence-split path. Token-aware validation is deferred to Milestone 3 when embeddings are generated.

**Rationale for character-based chunking at M2:** Token-aware chunking requires loading the MuRIL tokeniser, which is deferred to avoid a heavy preprocessing dependency. The character proxy is conservative and safe given the observed character-to-token ratio for this corpus.


In [11]:
print("\n" + "="*80)
print("STEP 9: CHUNK PREPARATION")
print("="*80)

CHUNK_SIZE = 512
OVERLAP = 50

print(f"Chunk Configuration:")
print(f"  Chunk size: {CHUNK_SIZE} characters")
print(f"  Overlap: {OVERLAP} characters")
print(f"  Format: Question: {{query}}\nAnswer: {{answer}}")
print(f"  Split Strategy: Sentence boundaries (.!?)")
print(f"  Target: {CHUNK_SIZE - 100}-{CHUNK_SIZE} chars (MuRIL limit)")

def create_chunks(row):
    """
    Create RAG chunks from Q&A pairs.

    Strategy:
    1. Combine Query and Answer with labels
    2. If text fits in CHUNK_SIZE, return single chunk
    3. If longer, split at sentence boundaries (.!?)
    4. Apply overlap between chunks to preserve context
    5. Each chunk carries full metadata for filtered retrieval
    """
    query = row.get('cleaned_query', '')
    answer = row.get('cleaned_answer', '')

    if not query and not answer:
        return []

    qa_text = f"Question: {query}\nAnswer: {answer}"

    # Single chunk - fits within limit
    if len(qa_text) <= CHUNK_SIZE:
        return [{
            'text': qa_text,
            'metadata': row.get('metadata', {}),
            'chunk_number': 1,
            'total_chunks': 1
        }]

    # Multiple chunks - split at sentence boundaries
    chunks = []
    sentences = re.split(r'(?<=[.!?])\s+', qa_text)
    current_chunk = ""
    chunk_num = 1

    for sentence in sentences:
        # If adding sentence exceeds CHUNK_SIZE, save current chunk
        if len(current_chunk) + len(sentence) + 1 > CHUNK_SIZE:
            if current_chunk:
                chunks.append({
                    'text': current_chunk.strip(),
                    'metadata': row.get('metadata', {}),
                    'chunk_number': chunk_num,
                    'total_chunks': 0
                })
                chunk_num += 1

                # Start new chunk with overlap (last 3-5 words)
                if OVERLAP > 0:
                    words = current_chunk.split()
                    if len(words) > 5:
                        overlap_text = ' '.join(words[-5:])
                        current_chunk = overlap_text + ' ' + sentence
                    elif len(words) > 3:
                        overlap_text = ' '.join(words[-3:])
                        current_chunk = overlap_text + ' ' + sentence
                    else:
                        current_chunk = sentence
                else:
                    current_chunk = sentence
            else:
                current_chunk = sentence
        else:
            current_chunk += ' ' + sentence if current_chunk else sentence

    # Add the last chunk
    if current_chunk:
        chunks.append({
            'text': current_chunk.strip(),
            'metadata': row.get('metadata', {}),
            'chunk_number': chunk_num,
            'total_chunks': 0
        })

    # Update total_chunks for all chunks
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

print("\nCreating chunks...")
all_chunks = []
processed = 0
failed = 0

batch_size = 10000
num_batches = (len(df_filtered) + batch_size - 1) // batch_size
print(f"Processing {len(df_filtered):,} records in {num_batches} batches...")

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(df_filtered))

    for idx in range(start_idx, end_idx):
        try:
            row = df_filtered.iloc[idx]
            chunks = create_chunks(row)
            if chunks:
                all_chunks.extend(chunks)
            processed += 1
        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f"  Error at row {idx}: {e}")

    if (batch_idx + 1) % 5 == 0:
        print(f"  Batch {batch_idx + 1}/{num_batches} ({end_idx:,} records) - Chunks: {len(all_chunks):,}")

print(f"\nChunking complete!")
print(f"  Records processed: {processed:,}")
print(f"  Chunks created: {len(all_chunks):,}")
print(f"  Failed: {failed}")

print("\nChunk statistics:")
if all_chunks:
    chunk_lengths = [len(c['text']) for c in all_chunks]
    print(f"  Total chunks: {len(all_chunks):,}")
    print(f"  Average length: {sum(chunk_lengths)/len(chunk_lengths):.0f} characters")
    print(f"  Min length: {min(chunk_lengths)}")
    print(f"  Max length: {max(chunk_lengths)}")

    single_chunk = sum(1 for c in all_chunks if c['total_chunks'] == 1)
    multi_chunk = len(all_chunks) - single_chunk
    print(f"  Single chunk records: {single_chunk:,} ({single_chunk/len(all_chunks)*100:.1f}%)")
    print(f"  Multi-chunk records: {multi_chunk:,} ({multi_chunk/len(all_chunks)*100:.1f}%)")


STEP 9: CHUNK PREPARATION
Chunk Configuration:
  Chunk size: 512 characters
  Overlap: 50 characters
  Format: Question: {query}
Answer: {answer}
  Split Strategy: Sentence boundaries (.!?)
  Target: 412-512 chars (MuRIL limit)

Creating chunks...
Processing 710,617 records in 72 batches...
  Batch 5/72 (50,000 records) - Chunks: 50,193
  Batch 10/72 (100,000 records) - Chunks: 100,394
  Batch 15/72 (150,000 records) - Chunks: 150,643
  Batch 20/72 (200,000 records) - Chunks: 200,998
  Batch 25/72 (250,000 records) - Chunks: 251,363
  Batch 30/72 (300,000 records) - Chunks: 301,833
  Batch 35/72 (350,000 records) - Chunks: 352,396
  Batch 40/72 (400,000 records) - Chunks: 402,966
  Batch 45/72 (450,000 records) - Chunks: 453,424
  Batch 50/72 (500,000 records) - Chunks: 503,844
  Batch 55/72 (550,000 records) - Chunks: 554,250
  Batch 60/72 (600,000 records) - Chunks: 604,708
  Batch 65/72 (650,000 records) - Chunks: 655,161
  Batch 70/72 (700,000 records) - Chunks: 705,580

Chunking 

## Step 9: Chunk Preview

Displays sample chunks and distribution statistics to verify output structure before writing to disk. Checks: chunk text format, metadata field completeness, chunk-per-record distribution, and crop-level coverage.


In [12]:
print("\n" + "="*80)
print("CHUNK PREVIEW")
print("="*80)

if len(all_chunks) > 0:

    print("\n1. SAMPLE CHUNKS:")
    print("-" * 60)

    for i, chunk in enumerate(all_chunks[:3], 1):
        print(f"\nChunk {i}:")
        print(f"  Crop: {chunk['metadata'].get('crop', 'N/A')}")
        print(f"  District: {chunk['metadata'].get('district', 'N/A')}")
        print(f"  Season: {chunk['metadata'].get('season', 'N/A')}")
        print(f"  Year: {chunk['metadata'].get('year', 'N/A')}")
        print(f"  Query Type: {chunk['metadata'].get('query_type', 'N/A')}")
        print(f"  Chunk: {chunk.get('chunk_number', 0)}/{chunk.get('total_chunks', 0)}")
        print(f"  Text: {chunk['text'][:200]}...")
        print("-" * 60)

    print("\n2. COMPLETE CHUNK (JSON):")
    print("-" * 60)
    print(json.dumps(all_chunks[0], indent=2, ensure_ascii=False))

    print("\n3. CHUNK DISTRIBUTION BY CROP:")
    print("-" * 60)
    crop_chunks = {}
    for chunk in all_chunks:
        crop = chunk['metadata'].get('crop', 'unknown')
        crop_chunks[crop] = crop_chunks.get(crop, 0) + 1

    for crop, count in sorted(crop_chunks.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(all_chunks)) * 100
        print(f"  {crop}: {count:,} chunks ({pct:.1f}%)")

    print("\n4. CHUNK DISTRIBUTION BY LANGUAGE:")
    print("-" * 60)
    lang_chunks = {}
    for chunk in all_chunks:
        lang = chunk['metadata'].get('language', 'unknown')
        lang_chunks[lang] = lang_chunks.get(lang, 0) + 1

    for lang, count in sorted(lang_chunks.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(all_chunks)) * 100
        print(f"  {lang}: {count:,} chunks ({pct:.1f}%)")

    print("\n5. CHUNK SIZE DISTRIBUTION:")
    print("-" * 60)
    chunk_lengths = [len(c['text']) for c in all_chunks]

    size_ranges = {
        '0-100': 0,
        '101-200': 0,
        '201-300': 0,
        '301-400': 0,
        '401-500': 0,
        '501-512': 0
    }

    for length in chunk_lengths:
        if length <= 100:
            size_ranges['0-100'] += 1
        elif length <= 200:
            size_ranges['101-200'] += 1
        elif length <= 300:
            size_ranges['201-300'] += 1
        elif length <= 400:
            size_ranges['301-400'] += 1
        elif length <= 500:
            size_ranges['401-500'] += 1
        else:
            size_ranges['501-512'] += 1

    for range_name, count in size_ranges.items():
        if count > 0:
            pct = (count / len(all_chunks)) * 100
            print(f"  {range_name} chars: {count:,} chunks ({pct:.1f}%)")

    print("\n6. CHUNK STATISTICS:")
    print("-" * 60)
    print(f"  Total chunks: {len(all_chunks):,}")
    print(f"  Average length: {sum(chunk_lengths)/len(chunk_lengths):.0f} characters")
    print(f"  Min length: {min(chunk_lengths)}")
    print(f"  Max length: {max(chunk_lengths)}")

    single_chunk = sum(1 for c in all_chunks if c['total_chunks'] == 1)
    multi_chunk = len(all_chunks) - single_chunk
    print(f"  Single chunk records: {single_chunk:,} ({single_chunk/len(all_chunks)*100:.1f}%)")
    print(f"  Multi-chunk records: {multi_chunk:,} ({multi_chunk/len(all_chunks)*100:.1f}%)")

    print("\n7. MULTI-CHUNK EXAMPLE:")
    print("-" * 60)
    multi_chunk_examples = [c for c in all_chunks if c['total_chunks'] > 1]
    if multi_chunk_examples:
        example = multi_chunk_examples[0]
        print(f"  Crop: {example['metadata'].get('crop', 'N/A')}")
        print(f"  Total chunks: {example['total_chunks']}")
        print(f"  Chunk {example['chunk_number']} of {example['total_chunks']}:")
        print(f"  Text: {example['text'][:150]}...")
    else:
        print("  No multi-chunk records found (all records fit in one chunk)")

else:
    print("No chunks created!")


CHUNK PREVIEW

1. SAMPLE CHUNKS:
------------------------------------------------------------

Chunk 1:
  Crop: Onion
  District: KANPUR CITY
  Season: Rabi
  Year: 2020
  Query Type: Plant Protection
  Chunk: 1/1
  Text: Question: Information related to preventing yellowing in onion nursery.....?
Answer: श्रीमान जी आप प्याज की नर्सरी में कार्बेंडाजिम 12 मैनकोज़ेब 63 डब्ल्यूपी दवा की 400 ग्राम मात्रा प्रति एकड़ की दर स...
------------------------------------------------------------

Chunk 2:
  Crop: Stevia
  District: SAHARANPUR
  Season: Rabi
  Year: 2020
  Query Type: Seeds and Planting Material
  Chunk: 1/1
  Text: Question: information about seed and planting material stevia ...................?
Answer: सर आप स्टीविया के पौधे प्राप्त करने के लिए सीमैप लखनऊ से संपर्क करें...
------------------------------------------------------------

Chunk 3:
  Crop: Lentil (Masur)
  District: MAHARAHGANJ
  Season: Rabi
  Year: 2020
  Query Type: Plant Protection
  Chunk: 1/1
  Text: Question: In

## Step 10: Save Artifacts

Writes all processed artifacts to the local repo under `../data/processed/kcc/` and `../data/final/kcc/`.

| Artifact | Directory | Size | Git-tracked | Contents |
|----------|-----------|------|-------------|----------|
| `kcc_cleaned_all_crops.csv` | `processed/kcc/` | 1.08 GB | No | Filtered, cleaned, deduplicated records with metadata column |
| `kcc_chunks_rag.jsonl` | `final/kcc/` | 674.0 MB | No | All chunks; one JSON object per line with `text` + `metadata` fields; primary input to MuRIL embedding in M3 |
| `kcc_chunks_sample_1000.jsonl` | `processed/kcc/` | 605.8 KB | No | 1,000-record random sample for fast pipeline testing |
| `metadata_schema.json` | `final/kcc/` | 8.3 KB | Yes | Chunk configuration, embedding model spec, metadata field definitions, RAG retrieval tier thresholds |
| `raw_kcc_sample.csv` | `sample/kcc/` | ~50 KB | **Yes** | 100-record stratified sample from the **raw** combined CSV (pre-preprocessing); exported by `03_kcc_rag_eda.ipynb` |
| `kcc_sample.csv` | `sample/kcc/` | ~50 KB | **Yes** | 100-record stratified, PII-redacted sample from the **processed** corpus; diff against `raw_kcc_sample.csv` to audit pipeline changes |

> **Repository policy (`data/sample/README.md`):** Large files (`raw/`, `processed/`, `final/`) are excluded by `.gitignore`. Both `data/sample/kcc/` files are tracked and together serve as a before/after verification pair for the preprocessing pipeline.


In [13]:
print("\n" + "="*80)
print("STEP 10: SAVING PROCESSED DATA")
print("="*80)

# Code using mounted Google Drive paths (commented out for colleague's convenience; uncomment to save to Drive)
# DRIVE_PROCESSED = f"{KCC_PATH}processed/"
# DRIVE_FINAL = f"{KCC_PATH}final/"

# Ensure local output directories exist
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)
Path(FINAL_PATH).mkdir(parents=True, exist_ok=True)

print(f"Output folders ready:")
print(f"  Processed: {PROCESSED_PATH}")
print(f"  Final:     {FINAL_PATH}")

# Helper function to convert numpy types to Python native types
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    else:
        return obj

print("\n" + "-"*60)
print("1. Saving Cleaned CSV")
print("-"*60)

cleaned_path_local = f"{PROCESSED_PATH}kcc_cleaned_all_crops.csv"
df_filtered.to_csv(cleaned_path_local, index=False)
print(f"  Saved: {cleaned_path_local}")
print(f"  Records: {len(df_filtered):,}")

# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to save to Drive)
# cleaned_path_drive = f"{DRIVE_PROCESSED}kcc_cleaned_all_crops.csv"
# df_filtered.to_csv(cleaned_path_drive, index=False)
# print(f"  Saved to Drive: {cleaned_path_drive}")

print("\n" + "-"*60)
print("2. Saving Chunks JSONL (Ready for MuRIL)")
print("-"*60)

chunks_path_local = f"{FINAL_PATH}kcc_chunks_rag.jsonl"
with open(chunks_path_local, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(convert_to_native(chunk), ensure_ascii=False) + '\n')
print(f"  Saved: {chunks_path_local}")
print(f"  Chunks: {len(all_chunks):,}")

# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to save to Drive)
# chunks_path_drive = f"{DRIVE_FINAL}kcc_chunks_rag.jsonl"
# with open(chunks_path_drive, 'w', encoding='utf-8') as f:
#     for chunk in all_chunks:
#         f.write(json.dumps(convert_to_native(chunk), ensure_ascii=False) + '\n')
# print(f"  Saved to Drive: {chunks_path_drive}")

print("\n" + "-"*60)
print("3. Saving Metadata Schema")
print("-"*60)

# Update schema with final counts
metadata_schema['total_records'] = int(len(df_filtered))
metadata_schema['total_chunks'] = int(len(all_chunks))
metadata_schema['language_distribution'] = convert_to_native(dict(df_filtered['query_lang'].value_counts()))
metadata_schema['crop_distribution'] = convert_to_native(dict(df_filtered['Crop'].value_counts()))
metadata_schema['query_type_distribution'] = convert_to_native(dict(df_filtered['QueryType'].value_counts().head(10)))

metadata_path_local = f"{FINAL_PATH}metadata_schema.json"
with open(metadata_path_local, 'w', encoding='utf-8') as f:
    json.dump(metadata_schema, f, indent=2, ensure_ascii=False)
print(f"  Saved: {metadata_path_local}")

# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to save to Drive)
# metadata_path_drive = f"{DRIVE_FINAL}metadata_schema.json"
# with open(metadata_path_drive, 'w', encoding='utf-8') as f:
#     json.dump(metadata_schema, f, indent=2, ensure_ascii=False)
# print(f"  Saved to Drive: {metadata_path_drive}")

print("\n" + "-"*60)
print("4. Saving Sample Chunks (For Quick Testing)")
print("-"*60)

sample_chunks = all_chunks[:1000]
sample_chunks_path_local = f"{PROCESSED_PATH}kcc_chunks_sample_1000.jsonl"
with open(sample_chunks_path_local, 'w', encoding='utf-8') as f:
    for chunk in sample_chunks:
        f.write(json.dumps(convert_to_native(chunk), ensure_ascii=False) + '\n')
print(f"  Saved: {sample_chunks_path_local}")

# Code using mounted Google Drive (commented out for colleague's convenience; uncomment to save to Drive)
# sample_chunks_path_drive = f"{DRIVE_PROCESSED}kcc_chunks_sample_1000.jsonl"
# with open(sample_chunks_path_drive, 'w', encoding='utf-8') as f:
#     for chunk in sample_chunks:
#         f.write(json.dumps(convert_to_native(chunk), ensure_ascii=False) + '\n')
# print(f"  Saved to Drive: {sample_chunks_path_drive}")

print("\n" + "-"*60)
print("5. Saving Q&A Pairs (For Easy Viewing)")
print("-"*60)

qa_cols = ['cleaned_query', 'cleaned_answer', 'Crop', 'DistrictName', 'QueryType', 'year', 'Season']
qa_cols = [col for col in qa_cols if col in df_filtered.columns]

if len(qa_cols) > 0:
    qa_df = df_filtered[qa_cols]
    qa_path_local = f"{PROCESSED_PATH}kcc_qa_pairs.csv"
    qa_df.to_csv(qa_path_local, index=False)
    print(f"  Saved: {qa_path_local}")
    print(f"  Records: {len(qa_df):,}")
    print(f"  Columns: {qa_df.columns.tolist()}")

    # Code using mounted Google Drive (commented out for colleague's convenience; uncomment to save to Drive)
    # qa_path_drive = f"{DRIVE_PROCESSED}kcc_qa_pairs.csv"
    # qa_df.to_csv(qa_path_drive, index=False)
    # print(f"  Saved to Drive: {qa_path_drive}")
else:
    print("  No Q&A columns available to save")

print("\n" + "="*60)
print("✅ All files saved successfully!")
print("="*60)


print("\n" + "-"*60)
print("6. Exporting verification sample  ->  data/sample/kcc/kcc_sample.csv")
print("-"*60)

SAMPLE_KCC_DIR = Path("../data/sample/kcc")
SAMPLE_KCC_PATH = SAMPLE_KCC_DIR / "kcc_sample.csv"
SAMPLE_SIZE = 100

# Drop unhashable columns (e.g. 'metadata' dict column) before deduplication
_sample_cols = [c for c in df_filtered.columns if c != 'metadata']
_df_for_sample = df_filtered[_sample_cols]

# Stratified: top-5 Categories x top-3 QueryTypes within each, capped at 100
_frames = []
_top_cats = _df_for_sample['Category'].value_counts().head(5).index.tolist()
for _cat in _top_cats:
    _cdf = _df_for_sample[_df_for_sample['Category'] == _cat]
    for _qt in _cdf['QueryType'].value_counts().head(3).index.tolist():
        _sub = _cdf[_cdf['QueryType'] == _qt]
        _n = max(1, SAMPLE_SIZE // (len(_top_cats) * 3))
        _frames.append(_sub.sample(min(_n, len(_sub)), random_state=42))

df_sample = pd.concat(_frames).drop_duplicates().head(SAMPLE_SIZE).copy()

# PII redaction — re-apply on all text columns present at this stage
def _redact(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\b\d{10}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\b[A-Z0-9]{8,}\b', '[ID]', text)
    return text

for _col in ['cleaned_query', 'cleaned_answer', 'QueryText', 'KccAns']:
    if _col in df_sample.columns:
        df_sample[_col] = df_sample[_col].apply(_redact)

SAMPLE_KCC_DIR.mkdir(parents=True, exist_ok=True)
df_sample.to_csv(SAMPLE_KCC_PATH, index=False)
print(f"  Saved: {SAMPLE_KCC_PATH}  ({len(df_sample)} records)")
print(f"  Category breakdown: {df_sample['Category'].value_counts().to_dict()}")

print("\n" + "="*60)
print("All artifacts saved.")
print("="*60)


STEP 10: SAVING PROCESSED DATA
Output folders ready:
  Processed: ../data/processed/kcc/
  Final:     ../data/final/kcc/

------------------------------------------------------------
1. Saving Cleaned CSV
------------------------------------------------------------
  Saved: ../data/processed/kcc/kcc_cleaned_all_crops.csv
  Records: 710,617

------------------------------------------------------------
2. Saving Chunks JSONL (Ready for MuRIL)
------------------------------------------------------------
  Saved: ../data/final/kcc/kcc_chunks_rag.jsonl
  Chunks: 716,287

------------------------------------------------------------
3. Saving Metadata Schema
------------------------------------------------------------
  Saved: ../data/final/kcc/metadata_schema.json

------------------------------------------------------------
4. Saving Sample Chunks (For Quick Testing)
------------------------------------------------------------
  Saved: ../data/processed/kcc/kcc_chunks_sample_1000.jsonl

--